In [2]:
import pandas as pd 
import yfinance as yf
import requests
from io import StringIO
import re

In [4]:
def load_nasdaq_tickers()-> pd.DataFrame:
    url = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt"
    response = requests.get(url, timeout=30)
    lines = response.text.strip().split("\n") 
    lines = lines[:-1]  # Remove the last line which is a footer
    df = pd.read_csv(StringIO("\n".join(lines)),sep="|")
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    print(len(df), "before filtering")
    df = df[df["test_issue"] == "N"]
    df = df[df["etf"] == "N"]
    print(len(df), "after etf/test filtering")
    exclude_pattern = re.compile(r"unit|right|warrant|trust|fund|etf|etn|depositary", re.IGNORECASE)
    df = df[~df["security_name"].str.contains(exclude_pattern, na=False)]
    print(len(df), "after name filtering")
    df = df[df["security_name"].str.contains("common stock", case=False, na=False)]
    print(len(df), "after common stock filtering")
    print(len(df[df["market_category"] == "Q"]), "after market category filtering")
    df['Exchange_QID'] = "Q82059"
    df["YahooTicker"] = df["symbol"]
    return df

load_nasdaq_tickers()

5496 before filtering
4265 after etf/test filtering
3256 after name filtering
2318 after common stock filtering
1063 after market category filtering


,symbol,security_name,market_category,test_issue,financial_status,round_lot_size,etf,nextshares,Exchange_QID,YahooTicker
16,AAL,"American Airlines Group, Inc. - Common Stock",Q,N,N,100,N,N,Q82059,AAL
18,AAME,Atlantic American Corporation - Common Stock,G,N,E,100,N,N,Q82059,AAME
19,AAOI,"Applied Optoelectronics, Inc. - Common Stock",G,N,N,100,N,N,Q82059,AAOI
20,AAON,"AAON, Inc. - Common Stock",Q,N,N,100,N,N,Q82059,AAON
24,AAPL,Apple Inc. - Common Stock,Q,N,N,40,N,N,Q82059,AAPL
...,...,...,...,...,...,...,...,...,...,...
5480,ZSTK,ZeroStack Corp. - Common Stock,S,N,N,100,N,N,Q82059,ZSTK
5481,ZTEK,Zentek Ltd. - common stock,S,N,D,100,N,N,Q82059,ZTEK
5487,ZUMZ,Zumiez Inc. - Common Stock,Q,N,N,100,N,N,Q82059,ZUMZ
5489,ZVRA,"Zevra Therapeutics, Inc. - Common Stock",Q,N,N,100,N,N,Q82059,ZVRA


In [5]:
def load_nyse_tickers() -> pd.DataFrame:
    url = "https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    lines = response.text.strip().split("\n") 
    lines = lines[:-1]  # Remove the last line which is a footer
    df = pd.read_csv(StringIO("\n".join(lines)),sep="|")
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    
    print(len(df), "before filtering")
    df = df[df["exchange"].isin(["N", "A", "P"])]
    print(len(df), "after filtering for exchange")
    df = df[df["etf"] == "N"]
    print(len(df), "after etf filtering")
    df = df[df["test_issue"] == "N"]
    print(len(df), "after test issue filtering")
    df = df[df["security_name"].str.contains("common stock", case=False, na=False)]
    print(len(df), "after common stock filtering")
    df["Exchange_QID"] = "Q13677"
    df['YahooTicker'] = df["cqs_symbol"]
    return df
load_nyse_tickers()

7304 before filtering
5902 after filtering for exchange
3188 after etf filtering
3169 after test issue filtering
1791 after common stock filtering


,act_symbol,security_name,exchange,cqs_symbol,etf,round_lot_size,test_issue,nasdaq_symbol,Exchange_QID,YahooTicker
0,A,"Agilent Technologies, Inc. Common Stock",N,A,N,100,N,A,Q13677,A
1,AA,Alcoa Corporation Common Stock,N,AA,N,100,N,AA,Q13677,AA
7,AADX,"Applied Aerospace & Defense, Inc. Common Stock",N,AADX,N,100,N,AADX,Q13677,AADX
8,AAMI,Acadian Asset Management Inc. Common Stock,N,AAMI,N,100,N,AAMI,Q13677,AAMI
16,AAT,"American Assets Trust, Inc. Common Stock",N,AAT,N,100,N,AAT,Q13677,AAT
...,...,...,...,...,...,...,...,...,...,...
7279,ZIP,"ZipRecruiter, Inc. Class A Common Stock",N,ZIP,N,100,N,ZIP,Q13677,ZIP
7288,ZONE,CleanCore Solutions Inc. Class B Common Stock,A,ZONE,N,100,N,ZONE,Q13677,ZONE
7298,ZTS,Zoetis Inc. Class A Common Stock,N,ZTS,N,100,N,ZTS,Q13677,ZTS
7299,ZVIA,Zevia PBC Class A Common Stock,N,ZVIA,N,100,N,ZVIA,Q13677,ZVIA


In [6]:
def load_hkex() -> pd.DataFrame:

    url = "https://www.hkex.com.hk/eng/services/trading/securities/securitieslists/ListOfSecurities.xlsx"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    print("Lade HKEX-Tickerliste herunter...")

    try:
        # Datei herunterladen
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # In Pandas einlesen (Die Daten starten meist ab Zeile 3)
        df = pd.read_excel(response.content, skiprows=2)
        # 1. Spalte als Text formatieren und mit Nullen auffüllen
        df["YahooTicker"] = df["Stock Code"].astype(str).str.split('.').str[0].str.zfill(4) + ".HK"

        # Spaltennamen säubern
        df.columns = [str(col).strip() for col in df.columns]

        # Relevante Spalten filtern (Stock Code und Name)
        # HKEX nutzt 5-stellige Nummern (z.B. 00001 statt 1)
        df = df[df["Sub-Category"] == "Equity Securities (Main Board)"]
        tickers = df[["Stock Code", "YahooTicker", "ISIN"]].dropna()
        tickers.rename(columns={"Stock Code": "OriginalTicker",}, inplace=True)
        tickers['Exchange_QID'] = "Q496672"
        print(f"Erfolgreich {len(tickers)} Ticker gefunden!")
        return tickers

    except Exception as e:
        print(f"Fehler beim Download: {e}")
df_hkex = load_hkex()
df_hkex

Lade HKEX-Tickerliste herunter...
Erfolgreich 2435 Ticker gefunden!


,OriginalTicker,YahooTicker,ISIN,Exchange_QID
0,1,0001.HK,KYG217651051,Q496672
1,2,0002.HK,HK0002007356,Q496672
2,3,0003.HK,HK0003000038,Q496672
3,4,0004.HK,HK0004000045,Q496672
4,5,0005.HK,GB0005405286,Q496672
...,...,...,...,...
17621,83690,83690.HK,KYG596691041,Q496672
17905,86618,86618.HK,KYG5074A1004,Q496672
17943,89618,89618.HK,KYG8208B1014,Q496672
17944,89888,89888.HK,KYG070341048,Q496672


In [7]:
def load_euronext_tickers() -> pd.DataFrame:
    url = "https://live.euronext.com/pd_es/data/stocks/download"
    params = {
        "mics": "dm_all_stock",
        "issueType": "101", # Common Stocks
        "initialLetter": "",
        "display_datapoints": (
            "logo,name,isin,symbol,market,"
            "lastPrice,precentDayChange,lastTradeTime"
        ),
        "fe_type": "csv",
        "fe_decimal_separator": ".",
        "fe_date_format": "d/m/Y",
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/csv",
    }

    # --- Download ---
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    csv_text = response.text

    # --- CSV korrekt einlesen ---
    df = pd.read_csv(
        StringIO(csv_text),
        sep=";", 
        encoding="utf-8-sig",
        low_memory=False,
    )
    df = df[3:] # Erste 3 Zeilen sind Metadaten

    # --- Erweitertes Mapping: Yahoo Suffix + Wikidata QID ---
    # Wir nutzen hier eine Struktur, die sowohl den Suffix als auch die WD-ID hält
    MARKET_CONFIG = {
        # Paris
        "Euronext Paris": {"suffix": ".PA", "qid": "Q2385849"},
        "Euronext Growth Paris": {"suffix": ".PA", "qid": "Q107188657"},
        "Euronext Access Paris": {"suffix": ".PA", "qid": "Q107190270"},
        # Amsterdam
        "Euronext Amsterdam": {"suffix": ".AS", "qid": "Q478720"},
        # Brussels
        "Euronext Brussels": {"suffix": ".BR", "qid": "Q1146518"}, # Gab zwei Ergebnisse
        "Euronext Growth Brussels": {"suffix": ".BR", "qid": "Q107189147"},
        "Euronext Access Brussels": {"suffix": ".BR", "qid": "Q107189150"},
        # Milan
        "Euronext Milan": {"suffix": ".MI", "qid": "Q936563"},
        "Euronext Growth Milan": {"suffix": ".MI", "qid": "Q23526757"},
        # Lisbon
        "Euronext Lisbon": {"suffix": ".LS", "qid": "Q2415561"},
        "Euronext Growth Lisbon": {"suffix": ".LS", "qid": "Q107189158"},
        "Euronext Access Lisbon": {"suffix": ".LS", "qid": "Q107188857"},
        # Dublin
        "Euronext Dublin": {"suffix": ".IR", "qid": "Q144458"}, #Gab zwei Ergebnisse
        "Euronext Growth Dublin": {"suffix": ".IR", "qid": "Q14Q107228013"},
        "Euronext Access Dublin": {"suffix": ".IR", "qid": "Q1435728"}, # Nicht gefunden
        # Oslo
        "Oslo Børs": {"suffix": ".OL", "qid": "Q909158"},
        "Euronext Growth Oslo": {"suffix": ".OL", "qid": "Q107190302"},
        "Euronext Expand Oslo": {"suffix": ".OL", "qid": "Q107188683"},
        # Multilistings
        "Euronext Brussels, Paris": {"suffix": ".BR", "qid": "Q2385845"},
        "Euronext Paris, Brussels": {"suffix": ".PA", "qid": "Q2385849"},
        "Euronext Brussels, Amsterdam": {"suffix": ".BR", "qid": "Q2385845"},
        "Euronext Amsterdam, Brussels": {"suffix": ".AS", "qid": "Q473938"},
        "Euronext Amsterdam, Paris": {"suffix": ".AS", "qid": "Q473938"},
        "Euronext Paris, Amsterdam": {"suffix": ".PA", "qid": "Q2385849"},
        "Euronext Paris, Amsterdam, Brussels": {"suffix": ".PA", "qid": "Q2385849"},
        "Euronext Amsterdam, Brussels, Paris": {"suffix": ".AS", "qid": "Q473938"},
        "Euronext Brussels, Amsterdam, Paris": {"suffix": ".BR", "qid": "Q2385845"},
        "Euronext Growth Brussels, Paris": {"suffix": ".BR", "qid": "Q2385845"},
        "Euronext Growth Paris, Brussels": {"suffix": ".PA", "qid": "Q2385849"},
    }
    # Nur relevante Märkte behalten
    df = df[df['Market'].isin(MARKET_CONFIG.keys())]

    # --- Spalten generieren ---
    # Yahoo Ticker
    df["YahooTicker"] = df.apply(
        lambda row: f"{row['Symbol']}{MARKET_CONFIG[row['Market']]['suffix']}", 
        axis=1
    )
    
    # Wikidata QID (Börsenplatz)
    df["Exchange_QID"] = df["Market"].apply(
        lambda x: MARKET_CONFIG[x]["qid"]
    )

    return df
df_euronext = load_euronext_tickers()
df_euronext

,Name,ISIN,Symbol,Market,Currency,Open Price,High Price,low Price,last Price,last Trade MIC Time,Time Zone,Volume,Turnover,Closing Price,Closing Price DateTime,YahooTicker,Exchange_QID
3,2020 BULKERS,BMG9156K1018,2020,Oslo Børs,NOK,4.32,4.34,3.902,3.96,05/06/2026 16:27,CET,460985,1861442.754,3.96,05/06/2026,2020.OL,Q909158
4,2CRSI,FR0013341781,AL2SI,Euronext Growth Paris,EUR,53.30,53.65,49.00,49.74,05/06/2026 17:39,CET,279221,14236284.235,49.74,05/06/2026,AL2SI.PA,Q107188657
11,4AIM SICAF,IT0005204729,AIM,Euronext Growth Milan,EUR,56.60,56.60,56.60,56.60,01/06/2026 10:55,CET,700,39620.00,56.60,05/06/2026,AIM.MI,Q23526757
12,4AIM SICAF COMP 2,IT0005440323,AIM2,Euronext Growth Milan,EUR,139.00,139.00,139.00,139.00,25/05/2026 10:08,CET,20,2780.00,139.00,05/06/2026,AIM2.MI,Q23526757
13,5TH PLANET GAMES,DK0060945467,5PG,Euronext Expand Oslo,NOK,1.36,1.44,1.36,1.41,05/06/2026 16:17,CET,208520,293227.37,1.41,05/06/2026,5PG.OL,Q107188683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3783,ZELLUNA,NO0013524942,ZLNA,Oslo Børs,NOK,22.80,23.00,22.40,22.90,05/06/2026 16:27,CET,8634,197052.60,22.90,05/06/2026,ZLNA.OL,Q909158
3784,ZENITH ENERGY,CA98936C8584,ZENA,Euronext Growth Oslo,NOK,0.656,0.658,0.617,0.65,05/06/2026 16:25,CET,2470412,1576389.761,0.65,05/06/2026,ZENA.OL,Q107190302
3785,ZEST,IT0005013013,ZEST,Euronext Milan,EUR,0.121,0.121,0.1175,0.1205,05/06/2026 17:11,CET,136645,16326.196,0.1205,05/06/2026,ZEST.MI,Q936563
3786,ZIGNAGO VETRO,IT0004171440,ZV,Euronext Milan,EUR,7.05,7.13,7.02,7.03,05/06/2026 17:35,CET,75756,535320.00,7.03,05/06/2026,ZV.MI,Q936563


In [8]:
df_euronext['Market'].value_counts()

Market
Euronext Paris                         305
Euronext Growth Paris                  254
Euronext Growth Milan                  205
Euronext Milan                         201
Oslo Børs                              198
Euronext Access Paris                  138
Euronext Amsterdam                     112
Euronext Brussels                       94
Euronext Growth Oslo                    85
Euronext Lisbon                         33
Euronext Dublin                         15
Euronext Access Lisbon                  14
Euronext Expand Oslo                     9
Euronext Access Brussels                 8
Euronext Growth Dublin                   7
Euronext Brussels, Paris                 6
Euronext Amsterdam, Brussels             5
Euronext Brussels, Amsterdam             5
Euronext Growth Brussels                 4
Euronext Paris, Brussels                 3
Euronext Paris, Amsterdam, Brussels      2
Euronext Amsterdam, Paris                1
Euronext Amsterdam, Brussels, Paris      1
Euro

In [9]:
nasdaq_t = load_nasdaq_tickers()[["symbol", "YahooTicker", "Exchange_QID"]]
print(len(nasdaq_t), "NASDAQ Tickers loaded")
nyse_t = load_nyse_tickers()[["cqs_symbol", "YahooTicker", "Exchange_QID"]]
euronext_t = load_euronext_tickers()[["Symbol", "YahooTicker", "Exchange_QID", "ISIN"]]
nasdaq_t.rename(columns={"symbol": "OriginalTicker"}, inplace=True)
nyse_t.rename(columns={"cqs_symbol": "OriginalTicker"}, inplace=True)
euronext_t.rename(columns={"Symbol": "OriginalTicker"}, inplace=True)

tickers =pd.concat([nasdaq_t, nyse_t, euronext_t,df_hkex], axis=0, ignore_index=True)

5496 before filtering
4265 after etf/test filtering
3256 after name filtering
2318 after common stock filtering
1063 after market category filtering
2318 NASDAQ Tickers loaded
7304 before filtering
5902 after filtering for exchange
3188 after etf filtering
3169 after test issue filtering
1791 after common stock filtering


In [10]:
tickers

,OriginalTicker,YahooTicker,Exchange_QID,ISIN
0,AAL,AAL,Q82059,NaN
1,AAME,AAME,Q82059,NaN
2,AAOI,AAOI,Q82059,NaN
3,AAON,AAON,Q82059,NaN
4,AAPL,AAPL,Q82059,NaN
...,...,...,...,...
8250,83690,83690.HK,Q496672,KYG596691041
8251,86618,86618.HK,Q496672,KYG5074A1004
8252,89618,89618.HK,Q496672,KYG8208B1014
8253,89888,89888.HK,Q496672,KYG070341048


In [11]:
tickers.to_csv("C:\\Diversification\\data\\tickers_08_06_2026.csv", index=False)